In [0]:
dbutils.widgets.text("bronze_layer", "bronze")
bronze = dbutils.widgets.get("bronze_layer")

dbutils.widgets.text("silver_layer", "silver")
silver = dbutils.widgets.get("silver_layer")

dbutils.widgets.text("schema_name", "fhir")
schema = dbutils.widgets.get("schema_name")

dbutils.widgets.text("tables", "Patient,Encounter,Observation,Condition")
tables_str = dbutils.widgets.get("tables")

Tables = [t.strip() for t in tables_str.split(",")]

In [0]:
table = None
for i in Tables:
    if i.lower() == "encounter":
        table = i
        break

if table is None:
    dbutils.notebook.exit("Exiting notebook: Table is not condition")

print(table + " table is present ")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Assigning the value to the variable and loading the data into df based on if the bronze table exist or not.

In [0]:
bronze_table = f"{bronze}.{schema}.{table}"
silver_table = f"{silver}.{schema}.{table}"

if spark.catalog.tableExists(silver_table):
    bronze_df = spark.table(bronze_table).filter(
        col("load_date") >= date_sub(current_date(), 4)
    )
else:
    bronze_df = spark.table(bronze_table)

bronze_df.select("load_date").show()

### Exploding and flattening the raw_JSON from df created from bronze table.

In [0]:

parsed_df = bronze_df.withColumn(
    "json_data",
    from_json(col("raw_json"), "array<struct<entry:array<struct<resource:struct<id:string,status:string,subject:struct<reference:string>,serviceProvider:struct<reference:string>,period:struct<start:string,end:string>,meta:struct<lastUpdated:string>>>>>>")
)

exploded_df = parsed_df.select(
    explode(col("json_data")).alias("bundle"),
    "extraction_timestamp"
)

entry_df = exploded_df.select(
    explode(col("bundle.entry")).alias("entry"),
    "extraction_timestamp"
)

In [0]:
staging_df = entry_df.select(
    col("entry.resource.id").alias("encounter_id"),
    col("entry.resource.status").alias("status"),
    col("entry.resource.subject.reference").alias("patient_ref"),
    col("entry.resource.serviceProvider.reference").alias("org_ref"),
    col("entry.resource.period.start").alias("period_start"),
    col("entry.resource.period.end").alias("period_end"),
    col("entry.resource.meta.lastUpdated").alias("last_updated")
)
staging_df.show(1)

In [0]:
staging_df = staging_df \
    .withColumn("patient_id", split(col("patient_ref"), "/")[1]) \
    .withColumn("organization_id", split(col("org_ref"), "/")[1]) \
    .drop("patient_ref", "org_ref")

In [0]:
staging_df = staging_df \
    .withColumn("effective_start_date", col("last_updated").cast("timestamp")) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

staging_df.orderBy("last_updated").show(1)

### Removing duplicate PK records based on last_updated date. So, only records having latest last_updated date will be considered.

In [0]:
window = Window.partitionBy("encounter_id").orderBy(col("last_updated").desc())

staging_df = staging_df \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

staging_df.show(1)

%md
### Checking if silver table exists or not if it exist than the the code will go to merge in case if silver table now exist df will directly created the table and exit the notebook

In [0]:

silver_table = f"{silver}.{schema}.{table}"

if not spark.catalog.tableExists(silver_table):
    staging_df.write.format("delta").saveAsTable(silver_table)
    dbutils.notebook.exit("Exiting notebook: As this was the first time load")

In [0]:
staging_df.createOrReplaceTempView("staging_encounter")

### SCD 2 Type loading data into silver table. Therefore, 3 things are handle - 
1. if an update is found for a record it will inactive the older record. 
2. if a new record is found it will insert it. 
3. and than it will insert the update record.

In [0]:

query = f"""
MERGE INTO {silver_table} AS tgt
USING staging_encounter AS src
ON tgt.encounter_id = src.encounter_id
AND tgt.is_current = true

WHEN MATCHED AND (
    tgt.status <> src.status OR
    tgt.patient_id <> src.patient_id OR
    tgt.organization_id <> src.organization_id OR
    tgt.period_start <> src.period_start OR
    tgt.period_end <> src.period_end
)
THEN UPDATE SET
    tgt.is_current = false,
    tgt.effective_end_date = current_timestamp()
"""

query1 = f"""
INSERT INTO {silver_table} (
    encounter_id,
    status,
    patient_id,
    organization_id,
    period_start,
    period_end,
    last_updated,
    effective_start_date,
    effective_end_date,
    is_current
)
SELECT 
    src.encounter_id,
    src.status,
    src.patient_id,
    src.organization_id,
    src.period_start,
    src.period_end,
    src.last_updated,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM staging_encounter src
JOIN {silver_table} tgt
    ON tgt.encounter_id = src.encounter_id
WHERE 
    tgt.is_current = false   -- was just inactivated
AND (
    tgt.status <> src.status OR
    tgt.patient_id <> src.patient_id OR
    tgt.organization_id <> src.organization_id OR
    tgt.period_start <> src.period_start OR
    tgt.period_end <> src.period_end
)
AND NOT EXISTS (
    SELECT 1
    FROM {silver_table} t2
    WHERE t2.encounter_id = src.encounter_id
      AND t2.is_current = true
)
"""

spark.sql(query)
spark.sql(query1)